<a href="https://colab.research.google.com/github/HafizaNoorUlSaba/langgraph/blob/main/Langgraph_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!pip install langchain langgraph langchain_core langchain-groq langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [1]:
from dotenv import load_dotenv

In [2]:
import os
from google.colab import userdata

# Retrieve the API key from Colab secrets and set it as an environment variable
# Ensure you have a secret named 'GROQ_API_KEY' in Colab secrets.
groq_api_key_from_secrets = userdata.get('Groq_Api_Key')

if groq_api_key_from_secrets:
    os.environ['GROQ_API_KEY'] = groq_api_key_from_secrets
    print("GROQ_API_KEY loaded from Colab secrets.")
else:
    print("Warning: GROQ_API_KEY secret not found in Colab secrets. Please ensure it's set correctly.")

GROQ_API_KEY loaded from Colab secrets.


In [3]:
from langchain_groq import ChatGroq
llm =ChatGroq(model_name="openai/gpt-oss-120b")

In [4]:
from langchain_core.messages import HumanMessage
messages = [HumanMessage("What is the weather today?")]

***Local Tools***

In [5]:
from langchain_core.tools import tool
from langchain_core.runnables import ConfigurableField
@tool
def add(x: int, y: int) -> int:
    """Adds two numbers and returns the sum."""
    return x + y
@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' times 'y'."""
    return x * y
@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the 'y'."""
    return x**y

In [6]:
tools = [multiply, exponentiate, add]
# Initialize the LLM with GPT-4o and bind the tools
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools(tools)

In [7]:
query = "What is 393 * 12.25? Also, what is 11 + 49?"
messages = [HumanMessage(query)]

In [8]:
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
  selected_tool = {"add": add, "multiply": multiply,
  "exponentiate": exponentiate}[tool_call["name"].lower()]
tool_msg = selected_tool.invoke(tool_call)
print(f'{tool_msg.name} {tool_call["args"]} {tool_msg.content}')
messages.append(tool_msg)
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

multiply {'x': 393, 'y': 12.25} 4814.25



In [9]:
# Get metadata for the Earth article on English Wikipedia
!curl 'https://en.wikipedia.org/api/rest_v1/page/summary/Earth'

{"type":"standard","title":"Earth","displaytitle":"<span lang=\"en\" dir=\"ltr\"><span class=\"mw-page-title-main\">Earth</span></span>","namespace":{"id":0,"text":""},"wikibase_item":"Q2","titles":{"canonical":"Earth","normalized":"Earth","display":"<span lang=\"en\" dir=\"ltr\"><span class=\"mw-page-title-main\">Earth</span></span>"},"pageid":9228,"thumbnail":{"source":"https://upload.wikimedia.org/wikipedia/commons/thumb/2/2d/Meteosat-12-fci-march-equinox-2025-noon.jpg/330px-Meteosat-12-fci-march-equinox-2025-noon.jpg","width":330,"height":330},"originalimage":{"source":"https://upload.wikimedia.org/wikipedia/commons/thumb/2/2d/Meteosat-12-fci-march-equinox-2025-noon.jpg/3840px-Meteosat-12-fci-march-equinox-2025-noon.jpg","width":12261,"height":12261},"lang":"en","dir":"ltr","revision":"1366037442","tid":"bfc4e10e-8878-11f1-b015-488908a8fa6e","timestamp":"2026-07-25T22:32:42Z","description":"Third planet from the Sun","description_source":"local","content_urls":{"desktop":{"page":"h

***API Based Tools***

In [11]:
!pip install wikipedia
from langchain_groq import ChatGroq
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage
api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=300)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)
# Initialize the LLM with GPT-4o and bind the tools
llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools([tool])
messages = [HumanMessage("What was the most impressive thing" +
"about Buzz Aldrin?")]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
  tool_msg = tool.invoke(tool_call)
print(tool_msg.name)
print(tool_call['args'])
print(tool_msg.content)
messages.append(tool_msg)
print()

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=67100e175079a278f326ac775b5f727d95b779691cde778c0dfdb932ef007224
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia
wikipedia
{'query': 'Buzz Aldrin'}
Page: Buzz Aldrin
Summary: Buzz Aldrin ( AWL-drin; born Edwin Eugene Aldrin Jr.; January 20, 1930) is an American former astronaut, aeronautical engineer, and fighter pilot. He was the second person to walk on the Moon after mission commander Neil Armstrong. He made three spacewalks as pilot of NASA



In [12]:
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

Buzz Aldrin’s most impressive achievement is that he was the **second human ever to set foot on the Moon**, a feat that combined his extraordinary technical expertise, piloting skill, and pioneering spirit.

### Why that stands out

| Aspect | What Aldrin did | Why it’s remarkable |
|--------|----------------|---------------------|
| **Technical mastery** | As a trained aeronautical engineer, Aldrin helped design the **space‑walk (EVA) procedures and hardware** that made the historic Moon walk possible. He also contributed to the development of the **Apollo Guidance Computer** and the **orbital rendezvous techniques** that are still the foundation of modern spaceflight. | He wasn’t just a “passenger” on the mission; he was a key engineer who turned the mission’s bold concepts into workable reality. |
| **Apollo 11 EVA** | On July 20 1969, after Neil Armstrong’s famous “one small step,” Aldrin descended the ladder of the Lunar Module *Eagle* and spent **21 minutes and 36 seconds walking

In [16]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.messages import HumanMessage
import requests

@tool
def get_stock_price(ticker: str) -> float:
  """Get the stock price for the stock exchange ticker for the company."""
  api_url = Stock_API_URL + ticker
  response = requests.get(api_url)
  if response.status_code == 200:
    data = response.json()
    return data["price"]
  else:
    raise ValueError(f"Failed to fetch stock price for {ticker}")

In [21]:
from unittest.mock import patch, Mock
import requests
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq
from langchain_core.tools import tool

# Define Stock_API_URL
Stock_API_URL = "https://api.example.com/stocks/"

# Redefine get_stock_price with mocked requests.get behavior
@tool
def get_stock_price(ticker: str) -> float:
  """Get the stock price for the stock exchange ticker for the company."""
  api_url = Stock_API_URL + ticker

  # Simulate an API response instead of making a real request to api.example.com
  if ticker.upper() == "AAPL":
    mock_response = Mock()
    mock_response.status_code = 200
    mock_response.json.return_value = {"price": 175.50}
    response = mock_response
  elif ticker.upper() == "GOOG":
    mock_response = Mock()
    mock_response.status_code = 200
    mock_response.json.return_value = {"price": 160.20}
    response = mock_response
  else:
    mock_response = Mock()
    mock_response.status_code = 404
    response = mock_response

  if response.status_code == 200:
    data = response.json()
    return data["price"]
  else:
    raise ValueError(f"Failed to fetch stock price for {ticker}")

llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=0)
llm_with_tools = llm.bind_tools([get_stock_price]) # Bind the newly defined get_stock_price
messages = [HumanMessage("What is the stock price of Apple?")]
ai_msg = llm_with_tools.invoke(messages)
messages.append(ai_msg)
for tool_call in ai_msg.tool_calls:
  tool_msg = get_stock_price.invoke(tool_call)
print(tool_msg.name)
print(tool_call['args'])
print(tool_msg.content)
messages.append(tool_msg)
print()

get_stock_price
{'ticker': 'AAPL'}
175.5



In [19]:
final_response = llm_with_tools.invoke(messages)
print(final_response.content)

Apple’s current stock price is **$175.50** per share.


***MCP Based Tools***

In [24]:
from typing import Sequence, Any, TypedDict
from langchain_core.tools import Tool # Corrected import path for Tool

# Placeholder for MultiServerMCPClient as it's not defined in the provided context
# You would need to define or import this class appropriately.
class MultiServerMCPClient:
    def __init__(self, config):
        self.config = config

    async def get_tools(self):
        # Simulate tool fetching based on config
        # In a real scenario, this would connect to your MCP servers
        mock_tools = []
        if "math" in self.config:
            mock_tools.append(Tool(name="math", func=lambda x: eval(x), description="Evaluates a mathematical expression."))
        if "weather" in self.config:
            mock_tools.append(Tool(name="weather", func=lambda x: "The weather is sunny.", description="Gets weather information."))
        return mock_tools

class AgentState(TypedDict):
  messages: Sequence[Any] # A list of BaseMessage/HumanMessage/...

mcp_client = MultiServerMCPClient(
  {
  "math": {
  "command": "python3",
  "args": ["src/common/mcp/MCP_weather_server.py"],
  "transport": "stdio", # Subprocess → STDIO JSON-RPC
},
  "weather": {

# Assumes a separate MCP server is already running on port 8000
"url": "http://localhost:8000/mcp",
"transport": "streamable_http",
# HTTP→JSON-RPC over WebSocket/stream
},
}
)

async def get_mcp_tools() -> list[Tool]:
  return await mcp_client.get_tools()

async def call_mcp_tools(state: AgentState) -> dict[str, Any]:
  messages = state["messages"]
  last_msg = messages[-1].content.lower()
  # Fetch and cache MCP tools on the first call
  global MCP_TOOLS
  if "MCP_TOOLS" not in globals():
    MCP_TOOLS = await mcp_client.get_tools()

  # Simple heuristic: if any digit-operator token appears, choose "math"
  if any(token in last_msg for token in ["+", "-", "*", "/", "(", ")"]):
    tool_name = "math"
  elif "weather" in last_msg:
    tool_name = "weather"
  else:
  # No match → respond directly
    return {
      "messages": [
        {
          "role": "assistant",
          "content": "Sorry, I can only answer math" +
          " or weather queries."
        }
      ]
    }

  tool_obj = next(t for t in MCP_TOOLS if t.name == tool_name)
  user_input = messages[-1].content
  mcp_result: str = await tool_obj.arun(user_input)
  return {
    "messages": [
      {"role": "assistant", "content": mcp_result}
    ]
  }